# 02 – Data Cleaning

**Goal:** turn `data/raw/` into a clean dataset in `data/processed/`.

Workflow:
1. Try each cleaning step here and check the result.
2. Copy the final steps into `ml/data/make_dataset.py` → the cleaning is **repeatable**.
3. Run `python -m ml.data.make_dataset` (or the last cell).

> ⚠️ Never edit the raw CSV by hand.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level down)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
raw = pd.read_csv(config.RAW_DATA_PATH)
df = raw.copy()
print(df.shape)

## Step 1 – Duplicates

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

## Step 2 – Inconsistent categories

In [ ]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

df["province"] = df["province"].replace({"phnom penh": "Phnom Penh", "PP": "Phnom Penh", "PHNOM PENH": "Phnom Penh",
                                         "siem reap": "Siem Reap", "Siemreap": "Siem Reap"})
df["payment_method"] = df["payment_method"].replace({"aba pay": "ABA Pay", "ABA": "ABA Pay", "wing": "Wing", "cash": "Cash"})
df["churn"] = df["churn"].replace({"yes": "Yes", "Y": "Yes", "no": "No", "N": "No"})

for col in ["province", "payment_method", "churn"]:
    print(df[col].value_counts(), "\n")

## Step 3 – Wrong data types and impossible values

In [ ]:
df["total_charges_usd"] = pd.to_numeric(df["total_charges_usd"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df.loc[~df["age"].between(16, 100), "age"] = np.nan
df.loc[~df["monthly_charges_usd"].between(0, 200), "monthly_charges_usd"] = np.nan
df["signup_date"] = pd.to_datetime(df["signup_date"])
df.dtypes

## Step 4 – Missing values

Decide for each column: **drop**, **fill**, or **let the model pipeline fill it**.

Here we leave them as `NaN`: the pipeline (`SimpleImputer` in `ml/features.py`) fills them
using the *training data only*, which avoids **data leakage**.

In [ ]:
df.isna().sum()[lambda s: s > 0]

## Step 5 – Compare with the reusable function in `ml/data/make_dataset.py`

In [ ]:
from ml.data.make_dataset import clean

clean_df = clean(raw)
print("Same result as this notebook:", clean_df.shape == df.shape)

## Save the processed data

In [ ]:
config.PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(config.PROCESSED_DATA_PATH, index=False)
print("Saved", config.PROCESSED_DATA_PATH)

## Cleaning log ✍️

👉 TODO: list every cleaning decision and **why** (this goes in your report).

| Problem | Rows affected | Decision | Reason |
|---|---|---|---|
| Duplicate rows | 35 | Removed | Same customer counted twice |
| Province spelling | ~150 | Mapped to standard names | Same place, different spelling |
| Impossible age | 6 | Set to missing → imputed | Data-entry error |
| … | | | |